# Wall design walkthrough — comparing symmetric and asymmetric shapes

This notebook walks through a real wall design problem:

1. Pick a target geometry (`lw`, `tw`, `hw`, materials).
2. Apply a fixed set of seismic demands (`WallDemands`).
3. Try **six configurations** that differ only in the boundary
   element (BE) topology — plain, barbell (small / large),
   C-shape, T-shape, Z-shape — and observe how the capacity,
   the BE checks (§18.10.6), and the proposed design change.

The demand will not necessarily pass for every configuration — the
point is to **see how the section responds** as the BE topology
evolves. This is the kind of trade-off study you would do at the
preliminary design stage before settling on a final geometry.

## API contract (see `design/AGENTS.md`)

```python
wall.run()           # lazy + cache
wall.capacity()      # WallCapacity
wall.check(demands)  # WallCheck
wall.design(demands) # WallDesignResults (minimum/demand/capacity/envelope)
wall.evolve(p)       # new Wall (immutable)
```

Internal units: **mm, MPa, kN, kN·mm** — the factory accepts metres
for `lu`/`hw` for back-compat and converts with a DeprecationWarning.

## 0. Setup

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt

from design import Concrete, Steel, Wall
from design.walls import WallDemands

# Silence the legacy-metres DeprecationWarning that the factory
# emits when lu/hw look like metres (lu < 30). We pass mm explicitly
# in this notebook so it shouldn't fire, but keep the filter for safety.
warnings.simplefilter('ignore', DeprecationWarning)

DISPLAY_UNITS = 5     # kN_mm_C — moments printed in kN·mm

## 1. Establish the demand state

The demands stay **fixed across every configuration**. They
represent the controlling load combo from the building analysis at
the wall's base (axial + in-plane bending + out-of-plane bending +
shear), plus the inelastic top displacement `delta_u` from a pushover
or DBD procedure, and the elastic compressive stress `sigma_max`
from a section-modulus check. The latter two drive the BE-required
tests in §18.10.6.2 and §18.10.6.3.

Values picked so that the **plain wall barely fails** but the
barbell variants pass with margin — that way the progression of
`ratio_PM` across configurations is informative. Internal units per
`AGENTS.md` §1: kN, mm, MPa, kN·m for moments.

| Variable | Value | Notes |
|---|---|---|
| `Pu` | 2 000 kN | factored axial at the base |
| `Mu` | 15 000 kN·m | in-plane, close to plain-wall capacity |
| `Mu_out` | 300 kN·m | out-of-plane (minor axis) |
| `Vu` | 1 800 kN | in-plane, close to `phi_Vn = 1850 kN` |
| `Vu_out` | 80 kN | out-of-plane |
| `delta_u` | 180 mm | `delta_u/hw = 0.006` → BE displacement check fires |
| `sigma_max` | 8 MPa | `> 0.20·f'c = 7 MPa` → BE stress check fires |

In [ ]:
demands = WallDemands(
    Pu=2000.0,        # kN
    Mu=15_000.0,      # kN·m in-plane
    Mu_out=300.0,     # kN·m out-of-plane
    Vu=1800.0,        # kN in-plane
    Vu_out=80.0,      # kN out-of-plane
    delta_u=180.0,    # mm top displacement (delta_u/hw = 0.006)
    sigma_max=8.0,    # MPa extreme fibre  (> 0.20·f'c)
)
demands

## 2. Base geometry — common to every configuration

Web length `lw = 4 000 mm`, web thickness `tw = 300 mm`, total
height `hw = 30 000 mm`, unbraced length `lu = 3 000 mm`. Concrete
`f'c = 35 MPa`, steel `fy = 420 MPa` (Grade 60). Web vertical bars
are ø12 distributed on the perimeter (14 along the length each
face); cover is 50 mm to the outside of the stirrup.

Every configuration uses the SAME web setup. Only the BE topology
(none / barbell / C / T / Z) and the BE size change.

In [ ]:
concrete = Concrete(fc=35)
steel    = Steel(fy=420, grade=60)

BASE = dict(
    lw=4000, tw=300, hw=30_000, lu=3_000,
    concrete=concrete, steel=steel,
    n_y_web=14, db_web=12, cover=50,
    n_x_be=4, n_y_be=5, db_be=22,    # column-style §18.10.6.4(b)
    db_stirrup=10, hx=200,
    seismic=True, omega_v_factor=1.5,
    units=DISPLAY_UNITS,
)
print(f'  lw = {BASE["lw"]} mm   tw = {BASE["tw"]} mm')
print(f'  hw = {BASE["hw"]} mm   lu = {BASE["lu"]} mm')
print(f'  fc = {concrete.fc} MPa   fy = {steel.fy} MPa')
print(f'  Web bars: {BASE["n_y_web"]} bars ø{BASE["db_web"]} per face')
print(f'  BE bars (when present): n_x_be={BASE["n_x_be"]} x n_y_be={BASE["n_y_be"]}, ø{BASE["db_be"]}')

## 3. Configuration #1 — Plain wall (no BE)

Start with the simplest case: a rectangular wall with distributed
vertical reinforcement only. This is the baseline — we expect the
displacement-based BE check (§18.10.6.2) and/or the stress-based
check (§18.10.6.3) to trip, which the design loop will respond to
by proposing BEs.

Use this as the *minimum* against which every other configuration
is judged.

In [ ]:
wall_1 = Wall.rectangular(**BASE, label='C1 plain')
wall_1.summary()

## 4. Configuration #2 — Small symmetric barbell

Add a modest barbell: `be_thickness = tw` (no protrusion in the
thickness direction) but a clear length `be_length = 400 mm` at
each end. This adds longitudinal steel at the extreme fibres
without changing the overall thickness — a common first step when
the BE check trips but space is tight.

Because `be_thickness == tw`, the BE is implicitly centred. This
is the cheapest BE you can add.

In [ ]:
wall_2 = Wall.rectangular(
    **BASE,
    be_top_length=400, be_top_thickness=300,    # = tw, no protrusion
    be_bot_length=400, be_bot_thickness=300,
    label='C2 small barbell',
)
wall_2.summary()

## 5. Configuration #3 — Large symmetric barbell

Increase to `be_thickness = 500 mm` (200 mm wider than the web)
and `be_length = 800 mm`. The BE now protrudes 100 mm on each
side. This is the classical 'good shear wall' geometry — symmetric
about both axes.

In [ ]:
wall_3 = Wall.rectangular(
    **BASE,
    be_top_length=800, be_top_thickness=500,
    be_bot_length=800, be_bot_thickness=500,
    label='C3 large barbell',
)
wall_3.summary()

## 6. Configuration #4 — C-shape (both BEs flush on the right side)

Same large BEs as configuration #3, but now `be_align='left'`:
both BEs are shifted so that their RIGHT face coincides with the
web's right face. The BEs protrude only to the LEFT.

This is what you build when the wall sits against a slab or core
that occupies the right side — you cannot let the BE protrude
there.

The section is still symmetric about the horizontal mid-plane
(`y = 0`), so the in-plane PM diagram stays symmetric (`Mn+ = Mn-`).
The geometric asymmetry shows up out-of-plane: the centre of mass
shifts to the left, giving more lever arm against `Mu_out` pushing
in one direction than the other.

In [ ]:
wall_4 = Wall.rectangular(
    **BASE,
    be_top_length=800, be_top_thickness=500,
    be_bot_length=800, be_bot_thickness=500,
    be_align='left',                # <- both BEs flush right
    label='C4 C-shape',
)
wall_4.summary()
s = wall_4.section
bb = s.bounding_box()
print()
print(f'  BE x-offset (top/bot) = {s.be_top_x_offset:+.0f} / {s.be_bot_x_offset:+.0f} mm')
print(f'  bbox x-range          = [{bb[0]:+.0f}, {bb[2]:+.0f}] mm (asymmetric)')
print(f'  shared face at x      = {s.tw/2:+.0f} mm (right side of the web)')

## 7. Configuration #5 — T-shape (top BE only)

Now drop the bottom BE entirely. The result is a T-shape: a
concentrated mass of steel and concrete at the top, plain web
below. This is what you get when a wall stops at a slab on top but
continues into a foundation that does NOT widen below.

Here the section is **truly asymmetric in-plane**: when the moment
puts the bottom in tension, you only have the distributed web bars
to resist. When the moment is reversed and the top is in tension,
you have the entire BE flange acting. Expect `Mn+ ≠ Mn-`.

In [ ]:
wall_5 = Wall.rectangular(
    **BASE,
    be_top_length=800, be_top_thickness=500,
    # be_bot_length = 0 (default)
    label='C5 T-shape',
)
wall_5.summary()

## 8. Configuration #6 — Z-shape (BEs flush to opposite sides)

Top BE flush left, bottom BE flush right (or vice versa). The
section loses both vertical and horizontal symmetry. This is the
most asymmetric topology in the catalogue.

Real-world example: a transfer wall where the boundary condition
changes between levels (e.g. perpendicular slab at the top floor
lands on the left, but at the foundation level the connection is
on the right because of a shear key).

In [ ]:
wall_6 = Wall.rectangular(
    **BASE,
    be_top_length=800, be_top_thickness=500,
    be_bot_length=800, be_bot_thickness=500,
    be_top_align='left',
    be_bot_align='right',           # <- opposite sides
    label='C6 Z-shape',
)
wall_6.summary()
print()
s = wall_6.section
print(f'  BE_top x-offset = {s.be_top_x_offset:+.0f} mm')
print(f'  BE_bot x-offset = {s.be_bot_x_offset:+.0f} mm')

## 9. Side-by-side — section geometry

Plot the six configurations on a single grid so the topology
differences are immediate.

In [ ]:
walls = [wall_1, wall_2, wall_3, wall_4, wall_5, wall_6]

fig, axes = plt.subplots(1, 6, figsize=(15, 8), sharey=True)
for ax, w in zip(axes, walls):
    w.plot.section(ax=ax)
    ax.set_title(w.label, fontsize=10)
    ax.axvline(0, color='gray', lw=0.4, ls=':')
    ax.set_xlim(-450, 450)         # uniform x-range so asymmetry is visible
plt.tight_layout()
plt.show()

## 10. Side-by-side — capacity

Run `capacity(biaxial=True)` on every wall and collect the
headline numbers. We expect:

- `Po` (pure compression) grows roughly with `Ag`.
- `phi_Vn` is fixed by the web (`Acv = tw·lw`) and barely
  changes between configurations.
- `max phi_Mn in-plane` jumps when BEs are added — they
  concentrate longitudinal steel at the extreme fibres.
- `max phi_Mn out-of-plane` is sensitive to BE protrusion in the
  thickness direction (configs #3, #4, #6).

In [ ]:
rows = []
for w in walls:
    cap = w.capacity(biaxial=True)
    Ag = w.section.gross_area()
    max_M_ip = float(np.max(cap.diagram.phi_Mn))
    min_M_ip = float(np.min(cap.diagram.phi_Mn))
    max_M_oop = float(np.max(cap.diagram_out.phi_Mn))
    rows.append([
        w.label, Ag / 1e6,         # m^2
        cap.Po, cap.To, cap.phi_Vn,
        max_M_ip, abs(min_M_ip), max_M_oop,
    ])

header = (f'{"config":<20s}  {"Ag (m^2)":>8s}  {"Po":>7s}  {"To":>7s}  '
          f'{"phi_Vn":>7s}  {"phi_Mn+ ip":>11s}  {"phi_Mn- ip":>11s}  {"phi_Mn oop":>11s}')
units  = (f'{"":<20s}  {"":>8s}  {"(kN)":>7s}  {"(kN)":>7s}  '
          f'{"(kN)":>7s}  {"(kN.m)":>11s}  {"(kN.m)":>11s}  {"(kN.m)":>11s}')
print(header)
print(units)
print('-' * len(header))
for r in rows:
    print(f'{r[0]:<20s}  {r[1]:>8.2f}  {r[2]:>7.0f}  {r[3]:>7.0f}  '
          f'{r[4]:>7.0f}  {r[5]:>11.0f}  {r[6]:>11.0f}  {r[7]:>11.0f}')

**Reading the table**

Look at the spread of `phi_Mn+ ip` vs `phi_Mn- ip` — for symmetric
configurations they're equal (rows 1, 2, 3, 4, 6). Only the T-shape
(row 5) shows a meaningful gap, because dropping the bottom BE
breaks the y-symmetry.

The out-of-plane column tells the opposite story: walls #3, #4
and #6 have larger BE protrusions in the thickness direction and
therefore higher out-of-plane capacity than #1 or #2.

## 11. Check vs the fixed demand

Run `check(demands)` on every wall and tabulate the ratios.
Reminder: failing isn't the point — we're observing trends.

In [ ]:
rows = []
for w in walls:
    chk = w.check(demands)
    rows.append([
        w.label,
        chk.ratio_pm, chk.ratio_shear,
        chk.be_required_disp, chk.be_required_stress,
        chk.be_length_required, chk.be_thickness_min,
        chk.distributed_rho_ok, chk.passed,
    ])

header = f'{"config":<20s}  {"ratio_PM":>9s}  {"ratio_V":>8s}  '\
         f'{"BE_disp":>8s}  {"BE_str":>7s}  {"BE_len":>7s}  {"BE_t":>6s}  '\
         f'{"rho_ok":>7s}  {"passed":>7s}'
print(header)
print('-' * len(header))
for r in rows:
    print(f'{r[0]:<20s}  {r[1]:>9.3f}  {r[2]:>8.3f}  '
          f'{str(r[3]):>8s}  {str(r[4]):>7s}  {r[5]:>7.0f}  {r[6]:>6.0f}  '
          f'{str(r[7]):>7s}  {str(r[8]):>7s}')

**Reading the check table**

- `ratio_PM` (demand / capacity at the PM point) drops dramatically
  once BEs are added — even small ones (config #2).
- `ratio_V` is nearly identical across configurations because
  `phi_Vn` is dominated by the web (`Acv = tw·lw`), which is
  unchanged.
- `BE_disp` (§18.10.6.2 displacement check) reports whether the
  inelastic top displacement requires a BE. For the slender wall
  with `delta_u/hw ≈ 0.004` this trips on the plain wall and
  configurations with very short BEs.
- `BE_str` (§18.10.6.3 stress check) trips when the extreme fibre
  compressive stress ≥ 0.20 f'c. With `sigma_max = 8 MPa < 0.20·35
  = 7 MPa`, this should fire across the board — observe.
- `BE_len` is the required BE length per §18.10.6.4(a) — useful as
  a target when sizing the BE for the next iteration.

## 12. What does `design()` propose for each?

Now call `wall.design(demands, auto_update=True)` on each
configuration. The auto-update loop will grow the BE iteratively
until the §18.10.6 demands are satisfied. We collect the envelope
of the proposed detailing.

Key insight: the auto-update **preserves the original alignment** —
a C-shape stays C-shape, a Z-shape stays Z-shape. The loop only
grows the BE length / thickness, never re-centres it.

In [ ]:
rows = []
for w in walls:
    res = w.design(demands, auto_update=True, max_iter=5)
    env = res.envelope
    final = res.final_wall
    rows.append([
        w.label,
        res.iterations, res.converged,
        env.be_required, env.be_length_proposed, env.be_thickness_proposed,
        final.section.be_top_x_offset, final.section.be_bot_x_offset,
        env.rho_t_required, env.Mpr_in_plane, env.Ve_capacity,
    ])

header = (f'{"config":<20s}  {"iters":>5s}  {"conv":>5s}  '
          f'{"BE_req":>7s}  {"BE_len":>7s}  {"BE_t":>6s}  '
          f'{"x_top":>6s}  {"x_bot":>6s}  {"rho_t":>7s}  '
          f'{"Mpr_ip":>9s}  {"Ve":>7s}')
units  = (f'{"":<20s}  {"":>5s}  {"":>5s}  {"":>7s}  '
          f'{"(mm)":>7s}  {"(mm)":>6s}  {"(mm)":>6s}  {"(mm)":>6s}  '
          f'{"":>7s}  {"(kN.m)":>9s}  {"(kN)":>7s}')
print(header)
print(units)
print('-' * len(header))
for r in rows:
    print(f'{r[0]:<20s}  {r[1]:>5d}  {str(r[2]):>5s}  '
          f'{str(r[3]):>7s}  {r[4]:>7.0f}  {r[5]:>6.0f}  '
          f'{r[6]:>+6.0f}  {r[7]:>+6.0f}  {r[8]:>7.4f}  '
          f'{r[9]:>9.0f}  {r[10]:>7.0f}')

**Reading the design table**

- `iters / conv`: most configurations converge in 1–2 iterations.
  Configurations that already had a BE long enough don't iterate.
- `BE_len / BE_t`: the proposed dimensions. For walls that
  started without BE (#1), this is the *new* BE the design wants
  to build.
- `x_top / x_bot`: the final offsets. **Note that the asymmetric
  configurations keep their offsets unchanged** through the
  iteration.
- `Mpr_ip`: probable moment (§18.10.3) — used to compute `Ve` via
  capacity-based shear (§18.10.3.1).
- `Ve`: the seismic shear the wall must resist by capacity design.
  Walls with higher `Mpr_ip` get higher `Ve`.

## 12b. Optimize sweep — which detailing weighs the least?

`wall.run_optimize(demands)` enumerates feasible combinations of

  * **web** distributed bars `(db_web, spacing, layers)`,
  * **BE longitudinal** bars `(db_be, n_y_be)` in the
    perimeter cage,

computes the steel quantity in kg/m³ of concrete for each,
and ranks them ascending by `rho_total`. The result mirrors
the ranking tables produced by `beam.run_optimize()` and
`column.run_optimize()`.

Important: the optimizer does **not** vary the BE geometry
(length, thickness, x-offset, n_x_be) — those are inputs you
fix when building the wall. To explore those, build several
walls and compare (that is exactly what configurations #1-#6
above are doing).

The table prints with one row per alternative:

```
rank  detail                         rho_t    rho_l    rho_tot   tag
                                    (kg/m3)  (kg/m3)  (kg/m3)
  1   web phi10@200x2c | BE 5xphi20   16.2    77.9      94.1
  2   web phi10@200x2c | BE 4xphi22   16.2    79.6      95.8
  ...
```

`rho_t` is the web horizontal (shear) reinforcement;
`rho_l` lumps the web vertical bars and the BE longitudinal
bars (which carry the moment). The baseline (the detailing
that `wall.design(demands)` would have used) is marked with
the `baseline` tag — `provided` is the as-built detailing,
and `infeasible` flags rows below `rho_min` or above the
spacing cap.

We run the sweep on configuration **#3 (large barbell)** —
it has BEs so the table shows both web and BE columns. For
the plain wall (#1) the BE columns disappear automatically.

In [ ]:
from design.walls.optimize import format_optimize_table

wall_to_optimize = wall_3            # large symmetric barbell
alts = wall_to_optimize.run_optimize(demands)
print(f'Alternatives explored : {len(alts)}')
print(f'Lightest rho_total    : {alts[0].rho_total:.1f} kg/m^3'
      f'   (detail: {alts[0].detailing})')
print()
print(format_optimize_table(alts, top_n=10))

Now run the same sweep on the **plain wall** (#1) — note
how the table shape contracts (no `BE` column) and the
ranges shrink because there is no BE longitudinal mass:

In [ ]:
alts_plain = wall_1.run_optimize(demands)
print(format_optimize_table(alts_plain, top_n=8))

And the **C-shape** (#4) — same web/BE structure as #3 but
with `be_align='left'`. Web sweep is identical (same `tw`),
and the BE sweep is identical (same length × thickness)
— the only difference is the geometric offset, which does
not affect the mass per unit volume. So the table should
match #3 numerically:

In [ ]:
alts_C = wall_4.run_optimize(demands)
print(format_optimize_table(alts_C, top_n=8))

**Reading the sweep**

  * The lightest BE detailing tends to combine a small
    diameter at fewer positions (`5×phi20` or `4×phi22`) —
    enough to clear the 1 % longitudinal floor of
    §18.10.6.4(b) without over-detailing.
  * Web `phi10@200x2c` (two curtains of phi10 at 200 mm)
    is the typical sweet spot for slender walls — it
    matches the §11.6 rho_min floor of 0.0025 and respects
    the §11.7.2 spacing limit `min(lw/5, 3·tw, 450) = 450`
    mm comfortably.
  * Going below the cheapest few rows usually means
    you tripped the BE longitudinal floor — those rows
    get marked `infeasible` and can be inspected with
    `wall.run_optimize(demands, include_infeasible=True)`.

## 13. PM diagrams — what's the same, what differs

Stack the in-plane PM diagrams of all six configurations. The
demand point (`Mu`, `Pu`) is plotted as a black dot.

Look for:
- The plain wall (#1) envelope is the smallest — it cannot host
  the demand point.
- Adding any BE expands the envelope rapidly.
- The T-shape (#5) is the only diagram that is asymmetric about
  `Mn = 0` (its left and right lobes differ).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
for w in walls:
    cap = w.capacity()
    Mn = np.asarray(cap.diagram.phi_Mn)        # kN.m
    Pn = np.asarray(cap.diagram.phi_Pn)        # kN
    ax.plot(Mn, Pn, label=w.label, lw=1.4)
ax.plot(demands.Mu, demands.Pu, 'ko', markersize=8,
        label=f'Demand (Mu={demands.Mu:.0f} kN.m, Pu={demands.Pu:.0f} kN)')
ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)
ax.set_xlabel('phi Mn  (kN.m)')
ax.set_ylabel('phi Pn  (kN)')
ax.set_title('In-plane PM envelopes - six wall configurations')
ax.legend(fontsize=9, loc='best')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 14. Out-of-plane PM — where the C / Z shapes win

Same exercise for the out-of-plane diagram (angle = 90°). The
demand is small (`Mu_out = 200 kN·m`) so every wall passes
comfortably; the interesting thing is the *envelope shape*.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
for w in walls:
    cap = w.capacity(biaxial=True)
    if cap.diagram_out is None:
        continue
    Mn = np.asarray(cap.diagram_out.phi_Mn)    # kN.m
    Pn = np.asarray(cap.diagram_out.phi_Pn)    # kN
    ax.plot(Mn, Pn, label=w.label, lw=1.4)
ax.plot(demands.Mu_out, demands.Pu, 'ko', markersize=8,
        label=f'Demand (Mu_out={demands.Mu_out:.0f} kN.m)')
ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)
ax.set_xlabel('phi Mn out-of-plane  (kN.m)')
ax.set_ylabel('phi Pn  (kN)')
ax.set_title('Out-of-plane PM envelopes - six wall configurations')
ax.legend(fontsize=9, loc='best')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Observation**: the asymmetric configurations (#4 C-shape, #6
Z-shape) have wider out-of-plane envelopes on one side because the
BE protrusion gives the section more lever arm against the
compression block. The plain wall (#1) and the small barbell (#2)
have the narrowest out-of-plane envelopes.

## 15. Take-aways

What this six-configuration sweep teaches:

1. **The web shear strength is set by the web** (`Acv = tw·lw`).
   Adding BEs barely moves `phi_Vn`. If shear governs, you have to
   widen the web — or accept that capacity-based `Ve` will exceed
   `phi_Vn` and detail accordingly.

2. **The PM envelope is dominated by where you put the
   longitudinal steel.** Distributed bars in the web are
   inefficient at large eccentricities; lumping bars into a BE at
   the extreme fibre roughly doubles the in-plane moment capacity
   for the same total `As`.

3. **Symmetric C-shape preserves in-plane symmetry.** If you flip
   both BEs to one side (e.g. against a slab), the in-plane
   response is the same as a barbell — you only pay the asymmetry
   out-of-plane.

4. **T-shape breaks in-plane symmetry.** `Mn+ != Mn-`, so the
   capacity-design shear (`Ve = (Mpr_pos + Mpr_neg) / l_n` for
   beams; `omega_v · Vu` for walls) and the BE detailing must be
   asymmetric too. The library handles this automatically because
   `wall.be_top` and `wall.be_bot` are independent `Column`
   instances.

5. **`wall.design(auto_update=True)` preserves the alignment.**
   The iterative BE-growth loop never re-centres an asymmetric BE;
   it just extends it along the wall length. This is what makes
   the C / T / Z shapes usable in production design.

## What's next

- Pick the best configuration for your geometric constraints.
- Run `wall.run_optimize(demands)` on it to explore web detailing
  alternatives ranked by kg/m³ of concrete.
- For the boundary elements, drill into `wall.be_top` (which is a
  full `Column` instance) and use its `column.run_optimize()` to
  pick `Ash` / hoop spacing / `db_hoop` / `n_legs`.
- Multi-flange topologies (I-shape, cruciform) are not yet
  supported by `Wall.rectangular()` — build them manually with
  `WallSection(...) + RebarLayout(groups=(...))` if needed.